###Machine Learning Workflow

1. Data Preparation & Feature EngineeringConvert your clinical and demographic columns into numerical formats suitable for algorithms:

Target Variable ($y$):readmitted: Binary (1 for Readmitted, 0 for Not Readmitted).

Demographic Features:age_group: One-Hot Encode or Ordinal Encode (Young Adult, Older Adults, Senior).gender: Binary/One-Hot Encode (Male, Female).Clinical 

Markers:avg_blood_sugar, blood_pressure_category, diagnoses_complexity (Low, Medium, High).length_of_stay (numerical feature).

In [0]:
# Imports & Data Loading
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Pull data directly from catalog
df = spark.table("patient_readmission.netcare_hospital.patient_readmission_csv").toPandas()
df.head()

In [0]:
# Define Features & Target matching your exact table column names
feature_cols = ['Age', 'Gender', 'Length of Stay', 'Blood Sugar Levels', 'Number of Diagnoses']

# Filter only columns present in df
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]

# Target column (Capital 'Readmission')
y = df['Readmission'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', 'true', '1'] else 0)

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")


Observations:

Feature Selection & Data Audit
What l did:

l isolated our target variable (Readmission) and selected 5 key clinical and demographic features (Age, Gender, Length of Stay, Blood Sugar Levels, and Number of Diagnoses) to feed into the model.

What the output means:

Dataset Dimensions (3000, 5): l successfully loaded all 3,000 patient records across our 5 feature columns.

Target Distribution:

2,134 patients (71%) were not readmitted (0).

866 patients (29%) were readmitted (1).

Takeaway: This confirms our data pipeline is reading directly from our source table and that our counts match the validated KPI dashboard figures exactly.

In [0]:
# Updated feature list with exact names
numeric_features = ['Age', 'Length of Stay', 'Blood Sugar Levels', 'Number of Diagnoses']
categorical_features = ['Gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

Feature Engineering & Data Preparation:

In this step, l organized our patient data and split it so l could train and test our machine learning model properly.

First, l separated our data types:

Numerical Features: Age, Length of Stay, Blood Sugar Levels, and Number of Diagnoses. l apply scaling to these so large numbers don't overwhelm smaller ones.

Categorical Features: Gender. l convert text categories into numbers so the model can process them mathematically.

Next, l created a 80/20 Train/Test Split:

80% (X_train / y_train): Used to train and teach the model patterns.

20% (X_test / y_test): Reserved as hidden, unseen patient records to evaluate how well the model predicts new, real-world cases.

Outcome: The output at the bottom shows X_test and X_train dataframes were successfully generated without errors, meaning our data pipeline is fully prepared for model training.

🔍 Code & Output Breakdown (In Simple Terms)
numeric_features & categorical_features:

What it does: Lists which columns contain numbers vs text categories.

preprocessor = ColumnTransformer(...):

What it does: Sets up automated cleaning rules. It scales numbers with StandardScaler() and converts text labels into binary flags with OneHotEncoder().

train_test_split(..., test_size=0.20, stratify=y):

What it does: Splitting 3,000 total patients into 2,400 training patients and 600 test patients.

stratify=y: Guarantees that both groups keep the exact same balance of readmitted vs non-readmitted patients (~29% readmissions).

Output (X_test and X_train previews at bottom):

What it means: Confirms that pandas DataFrames were created in memory with all 5 input fields ready for the training phase.

In [0]:

# Build & Train Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
print("Model training complete!")

Training the Model

What the code did: It fed the 2,400 training patient cases into a Random Forest algorithm (a collection of decision trees) so it could learn patterns linking length of stay, blood sugar, etc., to readmissions.

Result Explanation:

"Model training complete!" confirms the model successfully finished studying the training records without running into errors.

In [0]:

# Predict and Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred))

print("--- ROC-AUC SCORE ---")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

In this final step, I tested my model against 600 unseen patient records (the 20% test set) to see how accurately it predicts real-world readmissions.

Here is how the results break down:

Overall Accuracy (69%): At first glance, 69% accuracy looks decent. However, because 71% of the overall patients are not readmitted, the model gets high accuracy simply by guessing 'Not Readmitted' most of the time.

Readmission Precision (35%): When the model flags a patient as high-risk and predicts they will be readmitted (1), it is only correct 35% of the time.

Readmission Recall (11%): Out of 173 actual readmitted patients in my test set (support), the model only successfully caught 19 of them (11%). It missed 89% of high-risk patients.

ROC-AUC Score (0.4836): A score of 0.50 represents random guessing (like flipping a coin). A score of 0.4836 confirms my baseline model is currently not distinguishing readmitted patients from non-readmitted patients effectively.

Key Business Takeaway: The model is currently playing it 'too safe' because non-readmissions dominate the dataset. To make this a reliable decision tool for hospital staff, my immediate next step is to address class imbalance (by weighting readmission cases more heavily) and add richer clinical features.

In [0]:
#Logistic Regression with Balanced Class Weights

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

model.fit(X_train, y_train)

In [0]:
#XGBoost (Extreme Gradient Boosting)
%pip install xgboost

from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# Calculate ratio of non-readmitted to readmitted
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(scale_pos_weight=scale_weight, random_state=42))
])

model.fit(X_train, y_train)

In [0]:
#Random Forest with Class Weighting

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

model.fit(X_train, y_train)

In [0]:
from sklearn.ensemble import GradientBoostingClassifier

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

model.fit(X_train, y_train)

In [0]:
import joblib

# Save the trained model pipeline to the Databricks environment
joblib.dump(model, "readmission_pipeline.pkl")
print("Model successfully saved as readmission_pipeline.pkl!")

In [0]:
%pip install gradio

import gradio as gr
import pandas as pd

def predict_readmission(age, number_inpatient, time_in_hospital, num_lab_procedures):
    try:
        # 1. Prepare input DataFrame
        input_data = pd.DataFrame([{
            'age': float(age),
            'number_inpatient': float(number_inpatient),
            'time_in_hospital': float(time_in_hospital),
            'num_lab_procedures': float(num_lab_procedures)
        }])
        
        # 2. Use 'model' (matching Cell 15)
        prob_readmitted = model.predict_proba(input_data)[0][1]
        
        status = "⚠️ High Risk" if prob_readmitted >= 0.5 else "✅ Low Risk"
        prob_str = f"{prob_readmitted * 100:.1f}%"
        
        return status, prob_str
        
    except Exception as e:
        return f"Error: {str(e)}", "N/A"

demo = gr.Interface(
    fn=predict_readmission,
    inputs=[
        gr.Number(label="Age", value=65),
        gr.Number(label="Previous Inpatient Visits", value=1),
        gr.Slider(1, 14, value=4, step=1, label="Days in Hospital"),
        gr.Number(label="Lab Procedures", value=30)
    ],
    outputs=[
        gr.Textbox(label="Prediction Status"),
        gr.Textbox(label="Readmission Probability")
    ],
    title="🏥 Patient Readmission Risk Predictor"
)

demo.launch(share=True)

In [0]:
%pip install gradio

import gradio as gr
import pandas as pd

def predict_readmission(blood_sugar, num_diagnoses, gender, age, length_of_stay):
    try:
        # 1. Format inputs with correct data types
        input_data = pd.DataFrame([{
            'Blood Sugar Levels': float(blood_sugar),
            'Number of Diagnoses': int(num_diagnoses),
            'Gender': str(gender),
            'Age': int(age),
            'Length of Stay': int(length_of_stay)
        }])
        
        # 2. Get probability score from the pipeline
        prob_readmitted = model.predict_proba(input_data)[0][1]
        
        # 3. Categorize Risk Status based on clinical thresholds
        if prob_readmitted >= 0.5:
            status = "🚨 High Risk"
        elif prob_readmitted >= 0.3:
            status = "⚠️ Moderate Risk"
        else:
            status = "✅ Low Risk"
            
        prob_str = f"{prob_readmitted * 100:.1f}%"
        
        return status, prob_str
        
    except Exception as e:
        return f"Error: {str(e)}", "N/A"

# 4. Define Gradio Interface
demo = gr.Interface(
    fn=predict_readmission,
    inputs=[
        gr.Number(label="Blood Sugar Levels (mg/dL)", value=100),
        gr.Number(label="Number of Diagnoses", value=3),
        gr.Dropdown(["Male", "Female"], label="Gender", value="Female"),
        gr.Number(label="Age", value=65),
        gr.Slider(1, 14, value=4, step=1, label="Length of Stay")
    ],
    outputs=[
        gr.Textbox(label="Prediction Status"),
        gr.Textbox(label="Readmission Probability")
    ],
    title="🏥 Patient Readmission Risk Predictor"
)

demo.launch(share=True)